## **Customer Churn and Survival Analysis.**

## **Load data and drop leakage prone columns.**

In [1]:
import os
import pandas as pd
import numpy as np

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/CustomerChurn/"

data_df = pd.read_excel(DATASETS_PATH + "Telco_customer_churn.xlsx")

data_df = data_df.drop(columns=["Churn Score", "CLTV", "Churn Reason"])

print("Data: ")
print(data_df)


Data: 
      CustomerID  Count        Country       State          City  Zip Code  \
0     3668-QPYBK      1  United States  California   Los Angeles     90003   
1     9237-HQITU      1  United States  California   Los Angeles     90005   
2     9305-CDSKC      1  United States  California   Los Angeles     90006   
3     7892-POOKP      1  United States  California   Los Angeles     90010   
4     0280-XJGEX      1  United States  California   Los Angeles     90015   
...          ...    ...            ...         ...           ...       ...   
7038  2569-WGERO      1  United States  California       Landers     92285   
7039  6840-RESVB      1  United States  California      Adelanto     92301   
7040  2234-XADUH      1  United States  California         Amboy     92304   
7041  4801-JZAZL      1  United States  California  Angelus Oaks     92305   
7042  3186-AJIEK      1  United States  California  Apple Valley     92308   

                    Lat Long   Latitude   Longitude  Gen

## **Convert columns with string values that have more than one value to one-hot-encoding.**

In [2]:
data_df["Total Charges"] = pd.to_numeric(data_df["Total Charges"], errors="coerce")
data_df["Total Charges"] = data_df["Total Charges"].fillna(0.0)

data_df["Short_Term_Contract"] = (
    (data_df["Contract"] == "Month-to-month").astype(int)
)

data_df["Contract_Tenure_Risk"] = (
    data_df["Short_Term_Contract"] / (data_df["Tenure Months"] + 1)
)

risk_services = [
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support"
]

data_df["Missing_Services"] = (data_df[risk_services] == "No").sum(axis=1)

encoded_df = data_df.drop(columns=["Churn Value", "Churn Label", "CustomerID","Country","State","City","Lat Long", "Latitude", "Longitude", "Count", "Zip Code"])
y = data_df["Churn Value"]

for col in encoded_df.columns:
    if pd.api.types.is_string_dtype(data_df[col]):
        print(col)

Gender
Senior Citizen
Partner
Dependents
Phone Service
Multiple Lines
Internet Service
Online Security
Online Backup
Device Protection
Tech Support
Streaming TV
Streaming Movies
Contract
Paperless Billing
Payment Method


In [3]:
X_encoded = pd.get_dummies(encoded_df, drop_first=True)
print(X_encoded)

      Tenure Months  Monthly Charges  Total Charges  Short_Term_Contract  \
0                 2            53.85         108.15                    1   
1                 2            70.70         151.65                    1   
2                 8            99.65         820.50                    1   
3                28           104.80        3046.05                    1   
4                49           103.70        5036.30                    1   
...             ...              ...            ...                  ...   
7038             72            21.15        1419.40                    0   
7039             24            84.80        1990.50                    0   
7040             72           103.20        7362.90                    0   
7041             11            29.60         346.45                    1   
7042             66           105.65        6844.50                    0   

      Contract_Tenure_Risk  Missing_Services  Gender_Male  Senior Citizen_Yes  \
0     

In [4]:
print(X_encoded.columns)

Index(['Tenure Months', 'Monthly Charges', 'Total Charges',
       'Short_Term_Contract', 'Contract_Tenure_Risk', 'Missing_Services',
       'Gender_Male', 'Senior Citizen_Yes', 'Partner_Yes', 'Dependents_Yes',
       'Phone Service_Yes', 'Multiple Lines_No phone service',
       'Multiple Lines_Yes', 'Internet Service_Fiber optic',
       'Internet Service_No', 'Online Security_No internet service',
       'Online Security_Yes', 'Online Backup_No internet service',
       'Online Backup_Yes', 'Device Protection_No internet service',
       'Device Protection_Yes', 'Tech Support_No internet service',
       'Tech Support_Yes', 'Streaming TV_No internet service',
       'Streaming TV_Yes', 'Streaming Movies_No internet service',
       'Streaming Movies_Yes', 'Contract_One year', 'Contract_Two year',
       'Paperless Billing_Yes', 'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check'],
      dtype='str')


In [5]:
baseline_dict = {}

for col in encoded_df.select_dtypes(include=["string", "category"]).columns:
    categories = sorted(encoded_df[col].dropna().unique())
    baseline_dict[col] = categories[0]

print(baseline_dict)

{'Gender': 'Female', 'Senior Citizen': 'No', 'Partner': 'No', 'Dependents': 'No', 'Phone Service': 'No', 'Multiple Lines': 'No', 'Internet Service': 'DSL', 'Online Security': 'No', 'Online Backup': 'No', 'Device Protection': 'No', 'Tech Support': 'No', 'Streaming TV': 'No', 'Streaming Movies': 'No', 'Contract': 'Month-to-month', 'Paperless Billing': 'No', 'Payment Method': 'Bank transfer (automatic)'}


## **Divide into train and test datasets.**

In [6]:
from sklearn.model_selection import train_test_split

X_encoded["Zip Code"] = data_df["Zip Code"]

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

X_train = X_train.copy()
X_test = X_test.copy()

freq_map = X_train["Zip Code"].value_counts(normalize=True)

X_train["ZipCode_freq"] = X_train["Zip Code"].map(freq_map)
X_test["ZipCode_freq"] = X_test["Zip Code"].map(freq_map)

min_freq = freq_map.min()

X_test["ZipCode_freq"].fillna(min_freq)
X_train["ZipCode_freq"].fillna(min_freq)

X_train = X_train.drop(columns = "Zip Code")
X_test = X_test.drop(columns = "Zip Code")


## **Conduct logistic regression for baseline.**

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=100000)
model.fit(X_train, y_train)

y_pred_lr = model.predict(X_test)

y_proba_lr = model.predict_proba(X_test)[:, 1]

threshold_lr = 0.5

y_pred_lr = (y_proba_lr >= threshold_lr).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

from sklearn.metrics import roc_auc_score

print("ROC-AUC: ")
roc_auc_score(y_test, y_proba_lr)

              precision    recall  f1-score   support

           0       0.84      0.91      0.88      1009
           1       0.72      0.57      0.64       400

    accuracy                           0.81      1409
   macro avg       0.78      0.74      0.76      1409
weighted avg       0.81      0.81      0.81      1409

ROC-AUC: 


0.8598389494549059

## **Conduct classification with Random Forest.**

In [8]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.ensemble import RandomForestClassifier

import warnings
from sklearn.exceptions import FitFailedWarning, ConvergenceWarning

warnings.filterwarnings("ignore", category=FitFailedWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="One or more of the test scores are non-finite")

param_dist = {
    "n_estimators": [100, 200, 300, 500, 600, 700],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.5, 0.8],
    "bootstrap": [True],
    "max_samples": [0.5, 0.65, 0.8, None]
}

rf = RandomForestClassifier(
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

halving_search = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    scoring="roc_auc",
    cv=5,
    factor=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

halving_search.fit(X_train, y_train)

best_rf = halving_search.best_estimator_

print("Best params:", halving_search.best_params_)
print()

from sklearn.metrics import classification_report, roc_auc_score

y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))


n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 20
max_resources_: 5634
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 281
n_resources: 20
Fitting 5 folds for each of 281 candidates, totalling 1405 fits
----------
iter: 1
n_candidates: 94
n_resources: 60
Fitting 5 folds for each of 94 candidates, totalling 470 fits
----------
iter: 2
n_candidates: 32
n_resources: 180
Fitting 5 folds for each of 32 candidates, totalling 160 fits
----------
iter: 3
n_candidates: 11
n_resources: 540
Fitting 5 folds for each of 11 candidates, totalling 55 fits
----------
iter: 4
n_candidates: 4
n_resources: 1620
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 5
n_candidates: 2
n_resources: 4860
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best params: {'n_estimators': 600, 'min_samples_split': 20, 'min_samples_leaf': 4, 'max_samples': None, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}

   

## **Conduct classification with XGBoost model.**

In [9]:
param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 7, 10, 15],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.2, 0.5],
    "reg_lambda": [0.5, 1, 2, 5]
}

from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

halving_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    scoring="roc_auc",
    cv=5,
    factor=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

halving_search.fit(X_train, y_train)

best_xgb = halving_search.best_estimator_

print("Best params:", halving_search.best_params_)

y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))


n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 20
max_resources_: 5634
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 281
n_resources: 20
Fitting 5 folds for each of 281 candidates, totalling 1405 fits
----------
iter: 1
n_candidates: 94
n_resources: 60
Fitting 5 folds for each of 94 candidates, totalling 470 fits
----------
iter: 2
n_candidates: 32
n_resources: 180
Fitting 5 folds for each of 32 candidates, totalling 160 fits
----------
iter: 3
n_candidates: 11
n_resources: 540
Fitting 5 folds for each of 11 candidates, totalling 55 fits
----------
iter: 4
n_candidates: 4
n_resources: 1620
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 5
n_candidates: 2
n_resources: 4860
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best params: {'subsample': 0.6, 'reg_lambda': 0.5, 'n_estimators': 50, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.01, 'gamma': 0.2, 'colsample_bytree': 

## **Conduct classification with lightGBM model.**

In [10]:
from lightgbm import LGBMClassifier

param_dist = {
    "n_estimators": [10, 50, 100, 150, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "num_leaves": [7, 15, 31, 50, 70, 100],
    "max_depth": [-1, 1, 5, 10, 15],
    "min_child_samples": [5, 10, 20, 30, 50],
    "subsample": [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "reg_lambda": [0, 0.2, 0.5, 1, 2, 5]
}

lgb_model = LGBMClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

halving_search = HalvingRandomSearchCV(
    estimator=lgb_model,
    param_distributions=param_dist,
    scoring="roc_auc",
    cv=5,
    factor=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

halving_search.fit(X_train, y_train)

best_lgb = halving_search.best_estimator_

print("Best params:", halving_search.best_params_)

y_pred_lgb = best_lgb.predict(X_test)
y_proba_lgb = best_lgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgb))

n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 20
max_resources_: 5634
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 281
n_resources: 20
Fitting 5 folds for each of 281 candidates, totalling 1405 fits
----------
iter: 1
n_candidates: 94
n_resources: 60
Fitting 5 folds for each of 94 candidates, totalling 470 fits
----------
iter: 2
n_candidates: 32
n_resources: 180
Fitting 5 folds for each of 32 candidates, totalling 160 fits
----------
iter: 3
n_candidates: 11
n_resources: 540
Fitting 5 folds for each of 11 candidates, totalling 55 fits
----------
iter: 4
n_candidates: 4
n_resources: 1620
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 5
n_candidates: 2
n_resources: 4860
Fitting 5 folds for each of 2 candidates, totalling 10 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1469, number of negative: 4165
[LightGBM] [Info] A

## **Conclusion of Churn Probability Modeling:**
Several classification models were evaluated for customer churn prediction, including Logistic Regression, Random Forest, XGBoost, and LightGBM.

All models achieved similar **ROC-AUC scores (~0.86)**, indicating comparable ability to distinguish between churned and retained customers. However, significant differences were observed in precision-recall trade-offs.

Logistic Regression demonstrated higher precision but substantially lower recall for churned customers, meaning it failed to identify a large proportion of at-risk customers. In contrast, tree-based ensemble models (Random Forest, XGBoost, LightGBM) significantly improved recall, making them more suitable for churn detection tasks.

Among these, **LightGBM achieved the highest recall (0.81) for the churn class**, indicating the strongest ability to identify customers at risk of leaving. This comes at the cost of lower precision, implying more false positives.

From a business perspective, where missing a churner is typically more costly than incorrectly flagging a loyal customer, LightGBM provides the most effective model for churn prediction.

However, Logistic Regression remains valuable as a benchmark due to its interpretability and stable performance.

Further improvements could be achieved by optimizing the classification threshold to better align with business cost trade-offs.